In [1]:
import os  
import base64
from tqdm import tqdm
import time
from openai import AzureOpenAI  
from dotenv import load_dotenv
load_dotenv()

endpoint = os.getenv("AZURE_OPENAI_ENDPOINT_URL_1", "")  
deployment = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME_1", "")  
subscription_key = os.getenv("AZURE_OPENAI_API_KEY_1", "")  

# Initialize Azure OpenAI Service client with key-based authentication    
client = AzureOpenAI(  
    azure_endpoint=endpoint,  
    api_key=subscription_key,  
    api_version="2024-05-01-preview",
)

## Read validation data and label

In [10]:
import pandas as pd
validation_data= pd.read_csv("type_classification-validation.csv")

In [11]:
#Prepare the chat prompt 
new_validation_df = pd.DataFrame(columns=["Sentence", "Result"])

print("Running labelling")
for index, row in tqdm(validation_data.iterrows(), total=len(validation_data)):
    chat_prompt = [{
        "role": "system",
        "content": [
            {
                "type": "text",
                "text": "You are an AI Model that help to classify whether a sentence can be used to support information to create Class Diagram or Use Case Diagram or Activity Diagram\n\nBelow are given 5 sentences and whether it is useful for any of the diagram\n1. \"AP : As a case progresses , I need to record all the individuals and organizations that Verdict: take part in the case activities and the specific role they play .\"\nUseful for Class Diagram, Useful for Use Case Diagram, Useful for Activity Diagram\n\n2. \"Unless you are a celebrity or a good friend of Romano you will need a reservation .\"\nVerdict: Useful for Class Diagram, Useful for Use Case Diagram, Not Useful for Activity Diagram\n\n3. \"Therefore , there can be overlapping table reservations .\"\nVerdict: Useful for Class Diagram, Not Useful for Use Case Diagram, Not Useful for Activity Diagram\n\n4. \"These samples are sometimes sub - divided and distributed to multiple research teams or labs for different specialized observations .\"\nVerdict: Not Useful for Class Diagram, Useful for Use Case Diagram, Useful for Activity Diagram\n\n5. \"When the reservation party arrives at Romano 's the reservation is assigned to one waiter .\"\nVerdict: Useful for Class Diagram, Useful for Use Case Diagram, Not Useful for Activity Diagram\n\nUser will put a sentence and decide whether it will be useful for any of the category as an example output like this [Useful Class, Useful Use Case, Not Useful Activity]\n\n"
            }
        ]
    }, {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": row['sentence']
            }
        ]
    }]
    
    if index % 30 == 0 and index != 0:
        time.sleep(60)
        
    # Generate the completion  
    completion = client.chat.completions.create(  
        model=deployment,
        messages=chat_prompt,
        max_tokens=800,  
        temperature=0.7,  
        top_p=0.95,  
        frequency_penalty=0,  
        presence_penalty=0,
        stop=None,  
        stream=False
    )
    result = completion.choices[0].message.content
    new_validation_df = new_validation_df.append({"Sentence": row['sentence'], "Result": result}, ignore_index=True)
    
new_validation_df.to_csv("gpt4o-label-validation-5 example.csv")
    

Running labelling


  0%|          | 0/145 [00:00<?, ?it/s]C:\Users\RAYMOND\AppData\Local\Temp\ipykernel_11228\3921453607.py:40: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  new_validation_df = new_validation_df.append({"Sentence": row['sentence'], "Result": result}, ignore_index=True)
  1%|          | 1/145 [00:01<03:48,  1.59s/it]C:\Users\RAYMOND\AppData\Local\Temp\ipykernel_11228\3921453607.py:40: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  new_validation_df = new_validation_df.append({"Sentence": row['sentence'], "Result": result}, ignore_index=True)
  1%|▏         | 2/145 [00:02<02:28,  1.04s/it]C:\Users\RAYMOND\AppData\Local\Temp\ipykernel_11228\3921453607.py:40: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  new_validation_df = new_va

In [2]:
print("You are an AI Model that help to classify whether a sentence can be used to support information to create Class Diagram or Use Case Diagram or Activity Diagram\n\nBelow are given 5 sentences and whether it is useful for any of the diagram\n1. \"AP : As a case progresses , I need to record all the individuals and organizations that Verdict: take part in the case activities and the specific role they play .\"\nUseful for Class Diagram, Useful for Use Case Diagram, Useful for Activity Diagram\n\n2. \"Unless you are a celebrity or a good friend of Romano you will need a reservation .\"\nVerdict: Useful for Class Diagram, Useful for Use Case Diagram, Not Useful for Activity Diagram\n\n3. \"Therefore , there can be overlapping table reservations .\"\nVerdict: Useful for Class Diagram, Not Useful for Use Case Diagram, Not Useful for Activity Diagram\n\n4. \"These samples are sometimes sub - divided and distributed to multiple research teams or labs for different specialized observations .\"\nVerdict: Not Useful for Class Diagram, Useful for Use Case Diagram, Useful for Activity Diagram\n\n5. \"When the reservation party arrives at Romano 's the reservation is assigned to one waiter .\"\nVerdict: Useful for Class Diagram, Useful for Use Case Diagram, Not Useful for Activity Diagram\n\nUser will put a sentence and decide whether it will be useful for any of the category as an example output like this [Useful Class, Useful Use Case, Not Useful Activity]\n\n")

You are an AI Model that help to classify whether a sentence can be used to support information to create Class Diagram or Use Case Diagram or Activity Diagram

Below are given 5 sentences and whether it is useful for any of the diagram
1. "AP : As a case progresses , I need to record all the individuals and organizations that Verdict: take part in the case activities and the specific role they play ."
Useful for Class Diagram, Useful for Use Case Diagram, Useful for Activity Diagram

2. "Unless you are a celebrity or a good friend of Romano you will need a reservation ."
Verdict: Useful for Class Diagram, Useful for Use Case Diagram, Not Useful for Activity Diagram

3. "Therefore , there can be overlapping table reservations ."
Verdict: Useful for Class Diagram, Not Useful for Use Case Diagram, Not Useful for Activity Diagram

4. "These samples are sometimes sub - divided and distributed to multiple research teams or labs for different specialized observations ."
Verdict: Not Useful f

## Read Training data and label

In [2]:
import pandas as pd
training_data= pd.read_csv("type_classification-train.csv")

In [3]:
#Prepare the chat prompt 
new_training_df = pd.DataFrame(columns=["Sentence", "Result"])
data_count = len(training_data)

print("Running labelling")
for index, row in tqdm(training_data.iterrows(), total=len(training_data)):
    chat_prompt = [{
        "role": "system",
        "content": [
            {
                "type": "text",
                "text": "You are an AI Model that help to classify whether a sentence can be used to support information to create Class Diagram or Use Case Diagram or Activity Diagram\n\nBelow are given 5 sentences and whether it is useful for any of the diagram\n1. \"AP : As a case progresses , I need to record all the individuals and organizations that Verdict: take part in the case activities and the specific role they play .\"\nUseful for Class Diagram, Useful for Use Case Diagram, Useful for Activity Diagram\n\n2. \"Unless you are a celebrity or a good friend of Romano you will need a reservation .\"\nVerdict: Useful for Class Diagram, Useful for Use Case Diagram, Not Useful for Activity Diagram\n\n3. \"Therefore , there can be overlapping table reservations .\"\nVerdict: Useful for Class Diagram, Not Useful for Use Case Diagram, Not Useful for Activity Diagram\n\n4. \"These samples are sometimes sub - divided and distributed to multiple research teams or labs for different specialized observations .\"\nVerdict: Not Useful for Class Diagram, Useful for Use Case Diagram, Useful for Activity Diagram\n\n5. \"When the reservation party arrives at Romano 's the reservation is assigned to one waiter .\"\nVerdict: Useful for Class Diagram, Useful for Use Case Diagram, Not Useful for Activity Diagram\n\nUser will put a sentence and decide whether it will be useful for any of the category as an example output like this [Useful Class, Useful Use Case, Not Useful Activity]\n\n"
            }
        ]
    }, {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": row['sentence']
            }
        ]
    }]
    
    try:
        # Generate the completion  
        completion = client.chat.completions.create(  
            model=deployment,
            messages=chat_prompt,
            max_tokens=800,  
            temperature=0.7,  
            top_p=0.95,  
            frequency_penalty=0,  
            presence_penalty=0,
            stop=None,  
            stream=False
        )
        result = completion.choices[0].message.content
        new_training_df = new_training_df.append({"Sentence": row['sentence'], "Result": result}, ignore_index=True)
    except Exception as e:
        print(e)
        print("Error at index: ", index, " Sentence: ", row['sentence'])
        new_training_df = new_training_df.append({"Sentence": row['sentence'], "Result": ""}, ignore_index=True)
        
    
new_training_df.to_csv("gpt4o-label-training-5 example.csv")
    

Running labelling


  0%|          | 0/3130 [00:00<?, ?it/s]C:\Users\RAYMOND\AppData\Local\Temp\ipykernel_6272\4185285372.py:39: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  new_training_df = new_training_df.append({"Sentence": row['sentence'], "Result": result}, ignore_index=True)
  0%|          | 1/3130 [00:01<1:09:51,  1.34s/it]C:\Users\RAYMOND\AppData\Local\Temp\ipykernel_6272\4185285372.py:39: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  new_training_df = new_training_df.append({"Sentence": row['sentence'], "Result": result}, ignore_index=True)
  0%|          | 2/3130 [00:01<48:06,  1.08it/s]  C:\Users\RAYMOND\AppData\Local\Temp\ipykernel_6272\4185285372.py:39: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  new_training_df = new_training

Error code: 400 - {'error': {'message': "The response was filtered due to the prompt triggering Azure OpenAI's content management policy. Please modify your prompt and retry. To learn more about our content filtering policies please read our documentation: https://go.microsoft.com/fwlink/?linkid=2198766", 'type': None, 'param': 'prompt', 'code': 'content_filter', 'status': 400, 'innererror': {'code': 'ResponsibleAIPolicyViolation', 'content_filter_result': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': True, 'detected': True}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}}}}}
Error at index:  297  Sentence:  New . movement previous no was a there unvailable is action " undo "


C:\Users\RAYMOND\AppData\Local\Temp\ipykernel_6272\4185285372.py:39: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  new_training_df = new_training_df.append({"Sentence": row['sentence'], "Result": result}, ignore_index=True)
 10%|▉         | 299/3130 [50:11<4:37:41,  5.89s/it]C:\Users\RAYMOND\AppData\Local\Temp\ipykernel_6272\4185285372.py:39: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  new_training_df = new_training_df.append({"Sentence": row['sentence'], "Result": result}, ignore_index=True)
 10%|▉         | 300/3130 [50:19<5:08:07,  6.53s/it]C:\Users\RAYMOND\AppData\Local\Temp\ipykernel_6272\4185285372.py:39: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  new_training_df = new_training_df.append({"Sentence": row['sentenc

Error code: 400 - {'error': {'message': "The response was filtered due to the prompt triggering Azure OpenAI's content management policy. Please modify your prompt and retry. To learn more about our content filtering policies please read our documentation: https://go.microsoft.com/fwlink/?linkid=2198766", 'type': None, 'param': 'prompt', 'code': 'content_filter', 'status': 400, 'innererror': {'code': 'ResponsibleAIPolicyViolation', 'content_filter_result': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': True, 'detected': True}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}}}}}
Error at index:  545  Sentence:  When fact information that system system .


...

C:\Users\RAYMOND\AppData\Local\Temp\ipykernel_6272\4185285372.py:39: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  new_training_df = new_training_df.append({"Sentence": row['sentence'], "Result": result}, ignore_index=True)
 79%|███████▉  | 2487/3130 [7:03:46<1:39:48,  9.31s/it]C:\Users\RAYMOND\AppData\Local\Temp\ipykernel_6272\4185285372.py:39: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  new_training_df = new_training_df.append({"Sentence": row['sentence'], "Result": result}, ignore_index=True)
 79%|███████▉  | 2488/3130 [7:03:49<1:18:15,  7.31s/it]C:\Users\RAYMOND\AppData\Local\Temp\ipykernel_6272\4185285372.py:39: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  new_training_df = new_training_df.append({"Sentence": row['s

Error code: 400 - {'error': {'message': "The response was filtered due to the prompt triggering Azure OpenAI's content management policy. Please modify your prompt and retry. To learn more about our content filtering policies please read our documentation: https://go.microsoft.com/fwlink/?linkid=2198766", 'type': None, 'param': 'prompt', 'code': 'content_filter', 'status': 400, 'innererror': {'code': 'ResponsibleAIPolicyViolation', 'content_filter_result': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': True, 'detected': True}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}}}}}
Error at index:  2547  Sentence:  Every command must be acknowledged in a positive response startup order in a conclusion , the Gemini 8 m Telescopes


C:\Users\RAYMOND\AppData\Local\Temp\ipykernel_6272\4185285372.py:39: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  new_training_df = new_training_df.append({"Sentence": row['sentence'], "Result": result}, ignore_index=True)
 81%|████████▏ | 2549/3130 [7:14:13<1:09:03,  7.13s/it]C:\Users\RAYMOND\AppData\Local\Temp\ipykernel_6272\4185285372.py:39: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  new_training_df = new_training_df.append({"Sentence": row['sentence'], "Result": result}, ignore_index=True)
 81%|████████▏ | 2550/3130 [7:14:26<1:25:14,  8.82s/it]C:\Users\RAYMOND\AppData\Local\Temp\ipykernel_6272\4185285372.py:39: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  new_training_df = new_training_df.append({"Sentence": row['s

Error code: 400 - {'error': {'message': "The response was filtered due to the prompt triggering Azure OpenAI's content management policy. Please modify your prompt and retry. To learn more about our content filtering policies please read our documentation: https://go.microsoft.com/fwlink/?linkid=2198766", 'type': None, 'param': 'prompt', 'code': 'content_filter', 'status': 400, 'innererror': {'code': 'ResponsibleAIPolicyViolation', 'content_filter_result': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': True, 'detected': True}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}}}}}
Error at index:  2791  Sentence:  Nevertheless . system the in everything modify to able be to order prevent produced m may consequence direct a is which , context operational of the and requirements program Telescopes m 8 Gemini from speciﬁes the structure of complex a with c

C:\Users\RAYMOND\AppData\Local\Temp\ipykernel_6272\4185285372.py:39: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  new_training_df = new_training_df.append({"Sentence": row['sentence'], "Result": result}, ignore_index=True)
 89%|████████▉ | 2793/3130 [7:56:00<56:10, 10.00s/it]  C:\Users\RAYMOND\AppData\Local\Temp\ipykernel_6272\4185285372.py:39: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  new_training_df = new_training_df.append({"Sentence": row['sentence'], "Result": result}, ignore_index=True)
 89%|████████▉ | 2794/3130 [7:56:03<44:05,  7.87s/it]C:\Users\RAYMOND\AppData\Local\Temp\ipykernel_6272\4185285372.py:39: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  new_training_df = new_training_df.append({"Sentence": row['sen

In [4]:
print("done")

done


In [5]:
new_training_df

,Sentence,Result
0,This related to XML sources ; Process ANSI dat...,"[Useful Class, Useful Use Case, Useful Activity]"
1,"Establish rules , ; Namespace ; Provide a capa...","[Not Useful Class, Useful Use Case, Not Useful..."
2,This submitted ; Display an XML Tag Specificat...,"[Not Useful Class, Useful Use Case, Useful Act..."
3,Design ( models data to terms for the maintena...,"[Useful Class, Useful Use Case, Useful Activity]"
4,Content services ; Use ) related tags wizard X...,"[Not Useful Class, Not Useful Use Case, Not Us..."
...,...,...
3125,Each between a Channel Access client and an ar...,"[Useful Class, Not Useful Use Case, Not Useful..."
3126,Database,"[Not Useful Class, Not Useful Use Case, Not Us..."
3127,The heart of of transparent network provides w...,"[Useful Class, Not Useful Use Case, Not Useful..."
3128,"EPICS provides a software component , Channel ...","[Useful Class, Not Useful Use Case, Not Useful..."


In [2]:
print("You are an AI Model that help to classify whether a sentence can be used to support information to create Class Diagram or Use Case Diagram or Activity Diagram\n\nBelow are given 5 sentences and whether it is useful for any of the diagram\n1. \"AP : As a case progresses , I need to record all the individuals and organizations that Verdict: take part in the case activities and the specific role they play .\"\nUseful for Class Diagram, Useful for Use Case Diagram, Useful for Activity Diagram\n\n2. \"Unless you are a celebrity or a good friend of Romano you will need a reservation .\"\nVerdict: Useful for Class Diagram, Useful for Use Case Diagram, Not Useful for Activity Diagram\n\n3. \"Therefore , there can be overlapping table reservations .\"\nVerdict: Useful for Class Diagram, Not Useful for Use Case Diagram, Not Useful for Activity Diagram\n\n4. \"These samples are sometimes sub - divided and distributed to multiple research teams or labs for different specialized observations .\"\nVerdict: Not Useful for Class Diagram, Useful for Use Case Diagram, Useful for Activity Diagram\n\n5. \"When the reservation party arrives at Romano 's the reservation is assigned to one waiter .\"\nVerdict: Useful for Class Diagram, Useful for Use Case Diagram, Not Useful for Activity Diagram\n\nUser will put a sentence and decide whether it will be useful for any of the category as an example output like this [Useful Class, Useful Use Case, Not Useful Activity]\n\n")

You are an AI Model that help to classify whether a sentence can be used to support information to create Class Diagram or Use Case Diagram or Activity Diagram

Below are given 5 sentences and whether it is useful for any of the diagram
1. "AP : As a case progresses , I need to record all the individuals and organizations that Verdict: take part in the case activities and the specific role they play ."
Useful for Class Diagram, Useful for Use Case Diagram, Useful for Activity Diagram

2. "Unless you are a celebrity or a good friend of Romano you will need a reservation ."
Verdict: Useful for Class Diagram, Useful for Use Case Diagram, Not Useful for Activity Diagram

3. "Therefore , there can be overlapping table reservations ."
Verdict: Useful for Class Diagram, Not Useful for Use Case Diagram, Not Useful for Activity Diagram

4. "These samples are sometimes sub - divided and distributed to multiple research teams or labs for different specialized observations ."
Verdict: Not Useful f